## Trasformazioni di Householder

Le trasformazioni di Householder sono matrici simmetriche e ortogonali, utilizzate per ruotare un vettore su un asse cartesiano.

**Ricetta**: dato $a\in{\mathbb R}^n$, si definisce la matrice di Householder $U$ nel modo seguente

1. $\sigma = \|a\|$
2. $v = a + \sigma e_1$, dove $e_1$ è il primo vettore della base canonica
3. $\alpha =\frac 1 2 \|v\|^2$
4. $U = I-\frac{1}{\alpha}v v^T$

La matrice così costruita è ortogonale, simmetrica e ha la proprietà di ruotare il vettore $a$ portandolo lungo il primo asse cartesiano, ossia

$$Ua = -\sigma e_1 = \begin{pmatrix} -\|\sigma\|\\0\\\vdots\\ 0
\end{pmatrix} $$

In [2]:
n = 10
a = np.random.randn(n)

# =========================================
# Trasformazione di Householder (1ª riflessione)
# 
# Obiettivo: azzerare tutte le componenti di a
#            tranne la prima, preservando la norma
# =========================================

# 1) Norma del vettore originale
sigma = np.linalg.norm(a)
print('Norma vettore originale =', sigma)

# 2) Costruzione del vettore di Householder v
v = a.copy()

# Si usa sigma per costruire la riflessione stabile numericamente
# (evita cancellazione numerica)
v[0] += sigma

# v come vettore colonna
v = v.reshape(n, 1)

# 3) Costante di normalizzazione
alpha = (v.T @ v) / 2

# 4) Matrice di Householder
# H = I - 2 * (v v^T) / (v^T v)
H = np.eye(n) - (v @ v.T) / alpha

# =========================================
# Applicazione della trasformazione
# =========================================

print('Vettore riflesso =\n', H @ a)

# Output atteso:
# [ -||a||, 0, 0, ..., 0 ]
# (zeri numerici ~ precisione macchina)

# =========================================
# Proprietà della matrice di Householder
# =========================================

# Ortogonalità: H^T H = I
print('Ortogonalità:', np.linalg.norm(np.eye(n) - H @ H.T))

# Simmetria: H = H^T
print('Simmetria:', np.linalg.norm(H - H.T))

# Auto-inversa: H^-1 = H
print('Auto-inversa:', np.linalg.norm(H - np.linalg.inv(H)))

Norma vettore originale = 2.4447464848756013
Vettore riflesso =
 [-2.44474648e+00  9.71445147e-17 -6.24500451e-17  1.38777878e-17
  4.16333634e-17  1.11022302e-16  2.60208521e-18  2.77555756e-16
  1.38777878e-17 -4.44089210e-16]
Ortogonalità: 9.015272696053903e-16
Simmetria: 0.0
Auto-inversa: 8.277573980887406e-16


## Complessità computazionale effettiva nel calcolo del prodotto di un vettore per una matrice di Householder

Una volta definita una trasformazione di Householder associata ad un vettore $v$ con la formula $U=I-\frac{1}{\alpha}vv^T$, se si deve calcolare un prodotto $Uy$, dove $y$ è un qualsiasi vettore in $\mathbb R^n$, l'algoritmo più efficiente si ottiene applicando prima la proprietà distributiva e poi la associativa

$$Uy =\left(I-\frac{1}{\alpha}vv^T\right)y = y-\frac{1}{\alpha}v(v^Ty)  $$

INPUT: $v,y$
1. $\alpha \leftarrow \frac{1}{2} \|v\|^2$
2. $p \leftarrow v^Ty$
3. $z \leftarrow y - v(p/\alpha)$

OUTPUT: $z=Uy$

Calcolato in questo modo, il prodotto $Uy$ ha complessità proporzionale a $n$ invece che a $n^2$

## Stabilità delle fattorizzazioni: esperimento con la matrice di Wilkinson
Consideriamo la seguente matrice $n\times n$ (matrice di Wilkinson)
$$ A = \begin{pmatrix}
1 &   & & & & & 1\\
-1& 1 & & & & & 1\\
-1&-1& 1& & & & 1\\
\vdots & & &\ddots & & &\vdots\\
-1& -1 & -1 &        && 1& 1\\
-1& -1&-1 & \cdots & \cdots &-1&1
\end{pmatrix}$$
* Scrivere una function che, dato un intero $n$, restituisce la matrice di Wilkinson di dimensione $n$.
* Costruire un problema test relativo alla matrice di Wilkinson di dimensione 10 che abbia come soluzione il vettore di tutti 1
Calcolare la soluzione del sistema con la funzione linalg.solve e confrontare la soluzione calcolata con la soluzione esatta.
* Ripetere l'esperimento con $n=60$
* Ricalcolare la matrice con $n=10$, applicare la fattorizzazione $LU$ di Gauss e stampare la matrice $U$
* Risolvere il sistema lineare iniziale ($n=60$) mediante la fattorizzazione $QR$ e confrontare con la soluzione esatta


In [ ]:
import numpy as np
from scipy.linalg import lu, qr, solve_triangular

# 1) Matrice di Wilkinson
def wilkinson(n: int):
    # Triangolo inferiore con -1 sotto la diagonale
    M = np.tril(-np.ones((n, n)), -1)
    # Aggiungo la diagonale principale (identità)
    M += np.eye(n)
    # Modifico l'ultima colonna (eccetto ultima riga)
    M[:-1, n-1] = 1

    return M


# 2) Generazione del problema test
def gen_problem(n: int):
    W = wilkinson(n)     # matrice del sistema
    x = np.ones(n)       # soluzione esatta nota
    b = W @ x            # termine noto costruito a partire da x
    return W, b, x


# 3) Risoluzione e verifica errore con n=100
n = 100

W, b, x = gen_problem(n)

# Risoluzione del sistema lineare
x1 = np.linalg.solve(W, b)

# Soluzione numerica ottenuta
print('(n=100) Soluzione x =', x1)

# Errore rispetto alla soluzione esatta
print('(n=100) Errore relativo =', np.linalg.norm(x - x1))


# 4) Ricalcolare la matrice con n=10, applicare la fattorizzazione LU di Gauss e stampare la matrice U
n = 10

W, b, x = gen_problem(n)

# Fattorizzazione
PA, p = gauss_piv(W.copy())

# Estrazione L e U
U = np.triu(PA)
L = np.tril(PA, -1) + np.eye(n)

# Matrice di permutazione P
P = np.eye(n)[p]  # applica permutazione alle righe

# Stampa matrice U
print('(n=10) Matrice triangolare superiore U =\n', U)

# Verifica: PA ≈ LU
print("(n=10) Errore ||PA - LU|| =", np.linalg.norm(P @ W - L @ U))